In [ ]:
import os
from pprint import pprint

import pandas as pd
import numpy as np
import pyreadstat
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

In [ ]:
# --- Shared color palette (module 3.3) ---
PALETTE = ["#7BB3B2","#65A6BD","#C997AF","#B8B0D3","#F4CF97","#98B9A0","#F6DECD"]

# --- Plotly template: set once, every chart inherits it (module 3.10, Principle 5) ---
nso_template = go.layout.Template()
nso_template.layout = go.Layout(
    font=dict(family='Arial', size=13, color='#333'),
    title_font=dict(size=16, color='#222'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    colorway=PALETTE,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=60, r=30, t=60, b=40)
)
pio.templates['nso'] = nso_template
pio.templates.default = 'nso'

In [ ]:
# --- Plotly toolbar config ---
PLOTLY_CONFIG = {
    'displaylogo': False,
    'modeBarButtonsToRemove': ['select2d', 'lasso2d', 'autoScale2d'],
    'toImageButtonOptions': {'format': 'png', 'width': 1200, 'height': 700, 'scale': 2}
}

In [ ]:
# %% [Load data]
df, metadata = pyreadstat.read_sav('../../../data/0_raw/ZM_LFS_DATASET2024_Annual_10percent.sav')

In [ ]:
# %% [Inspect column labels]
print(set(metadata.column_labels))

In [ ]:
# %% [Select columns of interest]
# This cell was written after investigating the above result and picking
# column names that were likely to contain interesting data

cols = [
    # --- Demographics & Background ---
    "Is ... Male or Female?",
    "How old was ... at (his/her) last birthday?",
    "What is the highest grade/level of education that ... has successfully completed?",
    "What is ...'s current marital status?",
    "What is ...'s relationship to the head of the household?",
    "1. Province",
    "2. District",

    # --- Employment & Work ---
    "In the main job/business that (NAME) has, is she/he...",
    "INDUSTRY",
    "Occupation",
    "How many hours does (NAME) usually work per week in his/her...? Main job",
    "How many hours does (NAME) usually work per week in his/her...? OVERALL TOTAL",
    "What is the frequency of .....'s income/earnings in his/her main job?",
    "Would (NAME) want to work more hours per week than usually worked, provided the extra hours are paid?",
    "Is ?. employed on the basis of a written contract or an oral agreement?",

    # --- Income & Earnings ---
    "What is your annually/monthly/weekly/daily/hourly wage or salary before deductions?",
    "What are your annual/monthly/weekly/daily/hourly earnings after expenses?",
    "At what age did NAME start work for the first time in his /her life",

    # --- Time Use: Household Activities ---
    "During the last 7 days how much time did  (NAME) spend on Cleaning the house, washing clothes, cooking or shopping for the household",
    "During the last 7 days how much time did  (NAME) spend on Fetching water from natural or public sources for use by the household",
    "During the last 7 days how much time did (NAME) spend on Collecting firewood or other natural products for use as fuel by the household",
    "In the last 7 days, how much time did (NAME) spend on Leisure e.g., playing sports, watching TV etc.?",
    "In the last 7 days, how much time did (NAME) spend on Personal care e.g bathing, eating and sleeping?",
    "In the last 7 days, how much time did name spend travelling from home to\xa0place\xa0of\xa0work",

    # --- Time Use: Hours by Day of Week (Main Job) ---
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Monday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Tuesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Wednesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Thursday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Friday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Saturday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Sunday Main job?",

    # --- Financial Inclusion ---
    "P.20. Do you own a mobile phone",
    "P.21. Do you have a mobile money account in your own name",
    "P.23. How often do you use mobile money?",
    "P.25.  On a scale of 1 to 4, Do you find mobile money services to be cheap or expensive?",
    "P.30.A Savings at a bank",
    "P.30.G.Savings with savings group",
    "P.30.D.Savings that you keep on your mobile phone",
    "What method do you mainly use to pay for food/groceries?",

    # --- Education ---
    "Can... read and write in any language?",
    "Has... ever attended school?",
    "Is (NAME) currently attending school?",
    "Have (NAME) ever repeated any level of schooling any point in time?",
    "At what age did (NAME) begin school?",
]

# For each label we pick the original column name as it appears in df.
names_to_labels = metadata.column_names_to_labels
names_to_labels_reduced = {}
names = []
for col in cols:
    for name, label in names_to_labels.items():
        if label != col:
            continue
        names.append(name)
        names_to_labels_reduced[name] = label
pprint(names_to_labels_reduced)

In [ ]:
# %% [Filter dataframe to columns of interest]
df = df[names]

In [ ]:
df.groupby("A3")["A2"].mean().values

In [ ]:
# %% [Extract value labels from metadata]
variable_value_labels = metadata.variable_value_labels

In [ ]:
# %% [Chart 1: Age Distribution — Histogram]
# Pattern: Distribution (module 3.4, Pattern 1)
# Structure: single Histogram trace + layout (3.10 Principle 1)
# Reference line marks the median (module 3.6)

# ----- Step 1: Prepare -----
age = df["A3"].dropna()
median_age = age.median()

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Histogram(x=age, nbinsx=20, marker_color=PALETTE[0], name='Age')
)

fig.add_vline(
    x=median_age, line_dash='dash', line_color='red',
    annotation_text=f'Median: {median_age:.0f}',
    annotation_position='top right'
)

fig.update_layout(
    title='Age Distribution',
    xaxis_title='Age',
    yaxis_title='Count'
)
fig.show(config=PLOTLY_CONFIG)

# %% [Inspect gender value labels]
variable_value_labels["A2"]

In [ ]:
# %% [Chart 2: Age Pyramid — Two Horizontal Bar Traces]
# A population pyramid is two Bar traces going in opposite directions
# (3.10 Principle 2: multiple things on one chart = multiple traces)

# ----- Step 1: Prepare -----
bins = list(range(0, 85, 5))
age_labels = [f"{b} - {b+4}" for b in bins[:-1]]

df_pyr = df[["A2", "A3"]].dropna().copy()
df_pyr["age_group"] = pd.cut(df_pyr["A3"], bins=bins, labels=age_labels, right=False)
df_pyr["A2"] = df_pyr["A2"].map(variable_value_labels["A2"])

male = df_pyr[df_pyr["A2"] == 'Male'].groupby("age_group").size()
female = df_pyr[df_pyr["A2"] == 'Female'].groupby("age_group").size()

# ----- Step 2: Plot -----
fig = go.Figure()

fig.add_trace(go.Bar(
    y=age_labels, x=-male.reindex(age_labels, fill_value=0).values,
    name='Male', orientation='h', marker_color=PALETTE[0]
))
fig.add_trace(go.Bar(
    y=age_labels, x=female.reindex(age_labels, fill_value=0).values,
    name='Female', orientation='h', marker_color=PALETTE[2]
))

fig.update_layout(
    title='Age Pyramid',
    xaxis_title='Population Count',
    barmode='overlay',
    bargap=0.1,
    height=600
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 3: Population by Gender — Pie / Donut]
# Pie with donut hole (module 3.4: "use sparingly, 2-4 categories")
# Structure: single Pie trace + layout (3.10 Principle 1)

# ----- Step 1: Prepare -----
gender_counts = df['A2'].map(variable_value_labels["A2"]).value_counts()

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Pie(
        labels=gender_counts.index,
        values=gender_counts.values,
        hole=0.4,
        marker=dict(colors=PALETTE[:2], line=dict(color='white', width=1.5)),
        textinfo='percent+label'
    )
)

fig.update_layout(title='Population by Gender')
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 4: Highest Education Level — Horizontal Bar]
# Pattern: Comparison (module 3.4, Pattern 2)
# Horizontal bars are best for many categories / long labels

# ----- Step 1: Prepare -----
edu = df['B6'].map(variable_value_labels["B6"]).value_counts().sort_values()

# ----- Step 2: Plot (single horizontal Bar trace) -----
fig = go.Figure(
    go.Bar(
        y=edu.index, x=edu.values, orientation='h',
        marker_color=PALETTE[0],
        text=edu.values, textposition='outside'
    )
)

fig.update_layout(
    title='Highest Education Level Completed',
    xaxis_title='Number of respondents',
    height=max(400, len(edu) * 30)
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 5: Employment Overview — 2x2 Subplots]
# Mixed chart types in one figure: use make_subplots (module 3.6)
# Each subplot gets its own trace (3.10 Principle 2)

# ----- Step 1: Prepare -----
contract = df["D18"].map(variable_value_labels["D18"]).value_counts()
industry = df["INDUSTRY1"].map(variable_value_labels["INDUSTRY1"]).value_counts().head(10)
occupation = df["occupation1"].map(variable_value_labels["occupation1"]).value_counts().head(10)
wants_more = df["E5"].map(variable_value_labels["E5"]).value_counts()

# ----- Step 2: Plot -----
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Contract Type', 'Top 10 Industries',
                    'Top 10 Occupations', 'Wants More Work Hours?'),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "pie"}]]
)

fig.add_trace(
    go.Bar(y=contract.index, x=contract.values, orientation='h',
           marker_color=PALETTE[0], showlegend=False),
    row=1, col=1
)
fig.add_trace(
    go.Bar(y=industry.index, x=industry.values, orientation='h',
           marker_color=PALETTE[1], showlegend=False),
    row=1, col=2
)
fig.add_trace(
    go.Bar(y=occupation.index, x=occupation.values, orientation='h',
           marker_color=PALETTE[2], showlegend=False),
    row=2, col=1
)
fig.add_trace(
    go.Pie(labels=wants_more.index, values=wants_more.values,
           marker=dict(colors=PALETTE[:len(wants_more)]),
           textinfo='percent+label', showlegend=False),
    row=2, col=2
)

fig.update_layout(title='Employment Type Overview', height=700)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 6: Population by Province — Vertical Bar]
# Pattern: Comparison (module 3.4, Pattern 2)

# ----- Step 1: Prepare -----
prov = df['PROV'].value_counts().sort_values(ascending=False)

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Bar(
        x=prov.index.astype(str), y=prov.values,
        marker_color=PALETTE[0]
    )
)

fig.update_layout(
    title='Population by Province',
    yaxis_title='Count'
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 7: Hours Worked per Week — Histogram + Box Subplots]
# Two subplots: distribution + grouped comparison (modules 3.4 & 3.6)
# make_subplots for mixed chart types in one figure

# ----- Step 1: Prepare -----
hours = df["E3A"].dropna()
median_hours = hours.median()

gender_map = variable_value_labels["A2"]
df_hours = df[["A2", "E3A"]].dropna().copy()
df_hours["Gender"] = df_hours["A2"].map(gender_map)

# ----- Step 2: Plot -----
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribution', 'By Gender')
)

fig.add_trace(
    go.Histogram(x=hours, nbinsx=20, marker_color=PALETTE[0],
                 name='Hours', showlegend=False),
    row=1, col=1
)

for i, gender in enumerate(df_hours["Gender"].unique()):
    subset = df_hours[df_hours["Gender"] == gender]["E3A"]
    fig.add_trace(
        go.Box(y=subset, name=gender, marker_color=PALETTE[i]),
        row=1, col=2
    )

fig.add_vline(
    x=median_hours, line_dash='dash', line_color='red',
    annotation_text=f'Median: {median_hours:.0f}h',
    row=1, col=1
)

fig.update_layout(title='Hours Worked per Week \u2013 Main Job')
fig.update_xaxes(title_text='Hours per week', row=1, col=1)
fig.update_yaxes(title_text='Number of workers', row=1, col=1)
fig.update_yaxes(title_text='Hours per week', row=1, col=2)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 8: Time Use — Stacked Horizontal Bar]
# Pattern: Composition (module 3.4, stacked bar)
# One trace per activity, barmode='stack' (3.10 Principle 2)

# ----- Step 1: Prepare -----
time_cols = ['H8A', 'H8G', 'H8H', 'H8I', 'H8J']
time_labels = ['Household chores', 'Fetching water', 'Collecting firewood',
               'Leisure', 'Personal care']
avg = df[time_cols].apply(pd.to_numeric, errors='coerce').mean()

# ----- Step 2: Plot -----
fig = go.Figure()
for label, value, color in zip(time_labels, avg, PALETTE[:5]):
    fig.add_trace(
        go.Bar(y=['Average respondent'], x=[value], name=label,
               orientation='h', marker_color=color)
    )

fig.update_layout(
    title='Average time use in the last 7 days',
    xaxis_title='Hours',
    barmode='stack',
    height=300
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 9: Hours by Day x Gender — Heatmap]
# Pattern: Heatmap (module 3.4, Pattern 5)
# Single Heatmap trace (3.10 Principle 1)

# ----- Step 1: Prepare -----
day_cols = {
    'Monday': 'E1AC', 'Tuesday': 'E1AE', 'Wednesday': 'E1AG',
    'Thursday': 'E1AI', 'Friday': 'E1AK', 'Saturday': 'E1AM',
    'Sunday': 'E1AA'
}
gender_map = variable_value_labels.get('A2', {})

tmp = df[['A2', *day_cols.values()]].copy()
tmp = tmp.rename(columns={v: k for k, v in day_cols.items()})

for c in day_cols:
    tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
    tmp[c] = tmp[c].where(tmp[c].between(0, 24))

tmp['Gender'] = tmp['A2'].map(gender_map).fillna(tmp['A2'].astype(str))

long_df = tmp.melt(
    id_vars='Gender', value_vars=list(day_cols.keys()),
    var_name='Day', value_name='Hours'
).dropna(subset=['Hours'])

heatmap_data = long_df.pivot_table(
    index='Gender', columns='Day', values='Hours', aggfunc='mean'
).reindex(columns=list(day_cols.keys()))

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns.tolist(),
        y=heatmap_data.index.tolist(),
        text=heatmap_data.round(1).values,
        texttemplate='%{text:.1f}',
        colorscale='YlGnBu',
        colorbar=dict(title='Hours')
    )
)

fig.update_layout(
    title='Average main-job hours by day and gender',
    height=350
)
fig.show(config=PLOTLY_CONFIG)

# %% [Chart 10: Mobile Money Usage by Gender — Grouped Bar]
# Pattern: Comparison, grouped (module 3.4, Pattern 2)
# One trace per gender (3.10 Principle 2: multiple things = multiple traces)

# ----- Step 1: Prepare -----
def apply_labels(s, col):
    labels = variable_value_labels.get(col, {})
    return s.map(lambda x: labels.get(x, labels.get(int(x), labels.get(str(x), x))) if pd.notna(x) else x)

tmp = df[['A2', 'P_23_OFTEN']].dropna().copy()
tmp['Gender'] = apply_labels(tmp['A2'], 'A2')
tmp['Usage'] = apply_labels(tmp['P_23_OFTEN'], 'P_23_OFTEN')

order = [v for _, v in sorted(variable_value_labels.get('P_23_OFTEN', {}).items())]
ct = pd.crosstab(tmp['Usage'], tmp['Gender']).reindex(order).dropna(how='all')

# ----- Step 2: Plot -----
fig = go.Figure()
for i, gender in enumerate(ct.columns):
    fig.add_trace(
        go.Bar(x=ct.index, y=ct[gender], name=gender, marker_color=PALETTE[i])
    )

fig.update_layout(
    title='Mobile Money Usage by Gender',
    yaxis_title='Number of respondents',
    barmode='group'
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 11: Perceived Cost of Mobile Money — Diverging Bar]
# Diverging stacked bar: negative values pull left, positive pull right
# Multiple traces with barmode='relative' (3.10 Principles 2 & 3)

# ----- Step 1: Prepare -----
cost_col = 'P_25_FIND_MOBILE'
labels_map = variable_value_labels[cost_col]
order = sorted(labels_map)
cat_labels = [labels_map[k] for k in order]

pct = (
    df[cost_col].dropna()
    .value_counts(normalize=True)
    .reindex(order, fill_value=0)
    .mul(100)
)

diverging_colors = ['#d73027', '#fc8d59', '#91bfdb', '#4575b4']

# ----- Step 2: Plot -----
fig = go.Figure()
for i, (label, value, color) in enumerate(zip(cat_labels, pct, diverging_colors)):
    x_val = -value if i < 2 else value
    fig.add_trace(
        go.Bar(
            y=[' '], x=[x_val], name=label, orientation='h',
            marker_color=color,
            text=[f'{value:.0f}%' if value >= 4 else ''],
            textposition='inside', textfont=dict(color='white')
        )
    )

fig.add_vline(x=0, line_color='gray', line_width=1)
fig.update_layout(
    title='Perceived cost of mobile money services',
    xaxis_title='Share of respondents',
    barmode='relative',
    height=250
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 12: Savings Methods — Stacked Horizontal Bar]
# Pattern: Composition (module 3.4, stacked bar)
# Two traces: Yes + No (3.10 Principle 2)

# ----- Step 1: Prepare -----
savings_cols = ['P30A_SAVINGS_ATBANK', 'P30D_MOBILE_SAVINGS', 'P30G_SAVINGS_GROUP']

rows = []
for c in savings_cols:
    vlabels = variable_value_labels.get(c, {})
    s = df[c].map(vlabels).fillna(df[c]).astype(str).str.strip().str.lower()
    yes = s.str.contains('yes', na=False).sum()
    no = s.str.contains('no', na=False).sum()
    rows.append({'method': c.replace('P30', '').replace('_', ' ').title(), 'Yes': yes, 'No': no})

plot_df = pd.DataFrame(rows).set_index('method')
plot_df = plot_df.div(plot_df.sum(axis=1), axis=0) * 100

# ----- Step 2: Plot -----
fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=plot_df.index, x=plot_df['Yes'], name='Yes', orientation='h',
        marker_color=PALETTE[0],
        text=[f'{v:.0f}%' for v in plot_df['Yes']],
        textposition='inside', textfont=dict(color='white')
    )
)
fig.add_trace(
    go.Bar(
        y=plot_df.index, x=plot_df['No'], name='No', orientation='h',
        marker_color='#D3D3D3'
    )
)

fig.update_layout(
    title='Savings methods used',
    xaxis_title='Percent of respondents',
    xaxis_range=[0, 100],
    barmode='stack'
)
fig.show(config=PLOTLY_CONFIG)

In [ ]:
# %% [Chart 13: Hours Worked vs Weekly Wage — Scatter]
# Pattern: Relationship (module 3.4, Pattern 4)
# Single Scatter trace with mode='markers' (3.10 Principle 1)
# Log scale on y-axis for skewed wages

# ----- Step 1: Prepare -----
hours_col = 'E3A'
wage_col = 'FA3'
freq_col = 'FA1'

freq_labels = variable_value_labels.get(freq_col, {})
freq_text = df[freq_col].map(freq_labels).astype(str).str.lower()

day_cols_wage = ['E1AA', 'E1AC', 'E1AE', 'E1AG', 'E1AI', 'E1AK', 'E1AM']
worked_days = df[day_cols_wage].fillna(0).gt(0).sum(axis=1)

weekly_wage = np.select(
    [
        freq_text.str.contains('annual|year'),
        freq_text.str.contains('month'),
        freq_text.str.contains('week'),
        freq_text.str.contains('day|daily'),
        freq_text.str.contains('hour|hourly')
    ],
    [
        df[wage_col] / 52,
        df[wage_col] * 12 / 52,
        df[wage_col],
        df[wage_col] * worked_days,
        df[wage_col] * df[hours_col]
    ],
    default=np.nan
)

plot_df = df[[hours_col]].copy()
plot_df['weekly_wage'] = weekly_wage
plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna()
plot_df = plot_df[(plot_df[hours_col] > 0) & (plot_df['weekly_wage'] > 0)]

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Scatter(
        x=plot_df[hours_col], y=plot_df['weekly_wage'],
        mode='markers',
        marker=dict(color=PALETTE[0], size=6, opacity=0.45),
        name='Workers'
    )
)

fig.update_layout(
    title='Hours worked vs standardized weekly wage',
    xaxis_title='Usual hours worked per week',
    yaxis_title='Weekly wage (standardized)',
    yaxis_type='log'
)
fig.show(config=PLOTLY_CONFIG)